In [ ]:
import QuantLib as ql
import math
import pandas as pd
import random as rd
from py_vollib.black_scholes.implied_volatility import implied_volatility
from tqdm.notebook import tqdm, trange

In [ ]:
samples = 1000000
rd.seed(42)

In [ ]:
S0 = 1.0
q = 0.0
r = 0.0

option_data = list()
for i in trange(samples):
    
    sample_data = list()
    
    # Sample parameters
    eta = rd.uniform(0.0, 5.0)
    rho = rd.uniform(-1.0, 0.0)
    lamda = rd.uniform(0.0, 10.0)
    vbar = rd.uniform(0.0, 1.0)
    V0 = rd.uniform(0.0, 1.0)
    tau =  rd.uniform(0.001, 5)
    maturity = int(tau * 365)
    K = rd.uniform(0.5, 1.5)
    
    todaysDate = ql.Date(17, 2, 2024)
    ql.Settings.instance().evaluationDate = todaysDate

    settlementDate = todaysDate 
    day_count = ql.Actual365Fixed()
    rfr = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, r, day_count))
    div_yield = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, q, day_count))
    
    exercise = ql.EuropeanExercise(todaysDate + maturity)
    payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
    european_option = ql.VanillaOption(payoff, exercise)
        
    spot = ql.QuoteHandle(ql.SimpleQuote(S0))
    heston_process = ql.HestonProcess(rfr, div_yield, spot, V0, lamda, vbar, eta, rho)
    engine = ql.AnalyticHestonEngine(ql.HestonModel(heston_process), 1e-15, int(1e6))
    european_option.setPricingEngine(engine)
    
    try:
        price = european_option.NPV()
        if price > 0 and price + K >= S0:
            iv = implied_volatility(price, S0, K, tau, r, 'c')
            
            sample_data.append(eta)
            sample_data.append(rho)
            sample_data.append(lamda)
            sample_data.append(vbar)
            sample_data.append(V0)
            sample_data.append(tau)
            sample_data.append(K)
            sample_data.append(price)
            sample_data.append(iv)
            option_data.append(sample_data)
        
    except:
        # Ignore and go to the next sample
        continue 

In [ ]:
heading_list = ['eta', 'rho', 'lambda', 'vbar', 'V0', 'tau', 'K', 'price', 'iv']

In [ ]:
df = pd.DataFrame(option_data, columns=heading_list)

In [ ]:
df.head()

In [ ]:
df.to_csv('HestonData2.csv', index=False)